# A*, règles, contraintes et Bayes

Objectif : produire un chemin optimal, une preuve par règles, une affectation faisable et une probabilité conditionnelle. Données synthétiques, bibliothèque standard uniquement. Lire le cours du module 9 ; aucun téléchargement.

## 1. A* sur une grille

La file conserve g+h. Manhattan est adaptée aux quatre directions de coût unitaire. La table des coûts permet les améliorations.

In [1]:
import heapq
from itertools import product
size = 6
walls = {(2, 0), (2, 1), (2, 2), (2, 3), (4, 3)}
start, goal = (0, 0), (5, 5)
def astar(heuristic):
    queue = [(heuristic(start), 0, start)]
    costs, parents = {start: 0}, {}
    expanded = 0
    while queue:
        _, cost, node = heapq.heappop(queue)
        if cost != costs[node]:
            continue
        expanded += 1
        if node == goal:
            path = [node]
            while node in parents:
                node = parents[node]
                path.append(node)
            return cost, path[::-1], expanded
        for dx, dy in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            nxt = node[0] + dx, node[1] + dy
            if not (0 <= nxt[0] < size and 0 <= nxt[1] < size) or nxt in walls:
                continue
            proposal = cost + 1
            if proposal < costs.get(nxt, float('inf')):
                costs[nxt], parents[nxt] = proposal, node
                heapq.heappush(queue, (proposal + heuristic(nxt), proposal, nxt))
    return None, [], expanded
manhattan = lambda p: abs(p[0]-goal[0]) + abs(p[1]-goal[1])
cost, path, expanded = astar(manhattan)
reference = astar(lambda p: 0)
assert cost == reference[0] == len(path)-1
assert all(p not in walls for p in path)
assert all(abs(a[0]-b[0])+abs(a[1]-b[1]) == 1 for a,b in zip(path,path[1:]))
print({'coût A*': cost, 'états A*': expanded, 'états Dijkstra': reference[2], 'chemin': path})

{'coût A*': 10, 'états A*': 20, 'états Dijkstra': 25, 'chemin': [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (1, 5), (2, 5), (3, 5), (4, 5), (5, 5)]}


## 2. Chaînage avant et trace

Des faits nouveaux déclenchent les règles jusqu’au point fixe. La trace conserve les prémisses. L’absence de preuve ne signifie pas automatiquement fausseté.

In [2]:
facts = {'temperature_haute', 'capteur_valide'}
rules = [({'temperature_haute', 'capteur_valide'}, 'alerte'), ({'alerte'}, 'inspection')]
trace = []
changed = True
while changed:
    changed = False
    for premises, conclusion in rules:
        if premises <= facts and conclusion not in facts:
            facts.add(conclusion)
            trace.append((sorted(premises), conclusion))
            changed = True
assert 'inspection' in facts and 'panne' not in facts
print(trace)

[(['capteur_valide', 'temperature_haute'], 'alerte'), (['alerte'], 'inspection')]


## 3. CSP : détecter aussi l’impossibilité

Le backtracking teste uniquement les contraintes dont les variables sont affectées. Trois examens mutuellement incompatibles exigent trois créneaux.

In [3]:
def solve_csp(domains, constraints, assignment=None):
    assignment = {} if assignment is None else assignment
    if len(assignment) == len(domains):
        return assignment.copy()
    variable = min((v for v in domains if v not in assignment), key=lambda v: len(domains[v]))
    for value in domains[variable]:
        candidate = {**assignment, variable: value}
        if all(a not in candidate or b not in candidate or candidate[a] != candidate[b]
               for a,b in constraints):
            result = solve_csp(domains, constraints, candidate)
            if result is not None:
                return result
    return None
constraints = [('A','B'), ('B','C'), ('A','C')]
assert solve_csp({v: [0,1] for v in 'ABC'}, constraints) is None
solution = solve_csp({v: [0,1,2] for v in 'ABC'}, constraints)
assert all(solution[a] != solution[b] for a,b in constraints)
print(solution)

{'A': 0, 'B': 1, 'C': 2}


## 4. Réseau bayésien par énumération

Pluie et arrosage sont indépendants avant observation. On marginalise les configurations puis normalise. Le résultat attendu est calculé dans le cours.

In [4]:
wet = {(1,1): .99, (1,0): .9, (0,1): .8, (0,0): .01}
joint = {}
for rain, sprinkler, moist in product([0,1], repeat=3):
    p_r = .2 if rain else .8
    p_s = .3 if sprinkler else .7
    p_m = wet[rain,sprinkler] if moist else 1-wet[rain,sprinkler]
    joint[rain,sprinkler,moist] = p_r*p_s*p_m
assert abs(sum(joint.values())-1) < 1e-12
p_wet = sum(p for (r,s,m),p in joint.items() if m)
p_rain_wet = sum(p for (r,s,m),p in joint.items() if r and m)
posterior = p_rain_wet/p_wet
assert abs(p_wet-.383) < 1e-12
assert abs(posterior - .1854/.383) < 1e-12
print({'P(route mouillée)': p_wet, 'P(pluie | route mouillée)': posterior})

{'P(route mouillée)': 0.383, 'P(pluie | route mouillée)': 0.4840731070496084}


## Exercice

Remplacez Manhattan par zéro ; puis ajoutez une troisième couleur au CSP impossible. Pourquoi le graphe bayésien ne prouve-t-il pas un effet causal ?

In [5]:
# Écrivez votre expérience ici avant de lire la correction.

## Correction et limites

h=0 donne Dijkstra et le même coût optimal. Trois couleurs rendent le triangle satisfaisable, ce que contrôle déjà la cellule 3. Le réseau factorise une loi observationnelle ; une interprétation causale exige des hypothèses supplémentaires. Ces petits problèmes ne mesurent pas le passage à l’échelle.